# Deduplicate and Merge Similar Rules
This notebook demonstrates how to load a rules JSON file, find and merge similar rules using fuzzy string matching, and save a unique rule set back to JSON.

In [1]:
!pip install fuzzywuzzy python-Levenshtein

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Import required libraries
import json
import pandas as pd
from fuzzywuzzy import fuzz
from fuzzywuzzy import process


In [3]:
# Load rules from JSON file
rules_path = 'generated_rules.json'  # Update with your rules file path
with open(rules_path, 'r') as f:
    rules = json.load(f)

# Convert to DataFrame for easier processing
rules_df = pd.DataFrame(rules)
rules_df.head()

,rule_id,description,validation_criteria
0,R001,Register entries without clear triggers are li...,Check if entry lacking a trigger is classified...
1,R002,Entries without clear triggers indicate the ri...,Verify that entries without triggers have been...
2,R003,Register entries without clear triggers are no...,Confirm that entries lacking triggers cannot h...
3,R004,Entries without clear triggers may actually be...,Assess if entry without trigger describes some...
4,R005,Register entries without clear triggers may be...,Check if entry was inadequately identified dur...


In [4]:
# Find and group similar rules using fuzzy string matching
from collections import defaultdict

SIMILARITY_THRESHOLD = 85  # Adjust as needed
unique_groups = []
grouped = set()

for i, rule in rules_df.iterrows():
    if i in grouped:
        continue
    group = [i]
    desc1 = rule['description']
    for j, other_rule in rules_df.iterrows():
        if i == j or j in grouped:
            continue
        desc2 = other_rule['description']
        if fuzz.token_set_ratio(desc1, desc2) >= SIMILARITY_THRESHOLD:
            group.append(j)
            grouped.add(j)
    unique_groups.append(group)
    grouped.add(i)

# Merge grouped rules
merged_rules = []
for group in unique_groups:
    merged_desc = ' / '.join(set(rules_df.loc[group, 'description']))
    merged_criteria = ' / '.join(set(rules_df.loc[group, 'validation_criteria']))
    merged_rules.append({
        'description': merged_desc,
        'validation_criteria': merged_criteria
    })

print(f"Reduced from {len(rules_df)} to {len(merged_rules)} unique rules.")

Reduced from 39 to 39 unique rules.


In [5]:
# Save merged unique rules back to JSON
with open('unique_rules.json', 'w') as f:
    json.dump(merged_rules, f, indent=2)
print('Unique rules saved to unique_rules.json')

Unique rules saved to unique_rules.json
